# External validation and dataset shift

Exercise notebook for the lesson **[External validation and dataset shift](https://ramkumarmd.com/medical-ai/evaluation-safety/external-validation-dataset-shift)** in the *Evaluation, safety & deployment* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** External validation tests a model on data from a different hospital, population or period than the one it was built on. Dataset shift, such as different patients, a different lab assay or a different baseline risk, often leaves a model's ranking ability (AUROC) roughly intact while making its predicted risks wrong. Plan external validation with enough outcome events, report calibration as well as discrimination, and be ready to recalibrate or update the model for each new setting.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Simulating four settings

Develop at one synthetic site; test under three kinds of shift.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

predictors = ["age", "egfr", "hba1c", "albuminuria"]

def site(n, seed, age_mean=58, assay_bias=0.0, baseline=-2.0):
    """Synthetic cohort: 3-year risk of a kidney outcome. assay_bias shifts measured eGFR only."""
    rng = np.random.default_rng(seed)
    d = pd.DataFrame({"age": rng.normal(age_mean, 10, n).round(),
                      "true_egfr": rng.normal(70, 20, n).clip(15, 120),
                      "hba1c": rng.normal(7.8, 1.3, n).round(1),
                      "albuminuria": rng.binomial(1, 0.3, n)})
    logit = (baseline + 0.02 * (d.age - 58) - 0.04 * (d.true_egfr - 70)
             + 0.3 * (d.hba1c - 7.8) + 1.0 * d.albuminuria)
    d["outcome"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    d["egfr"] = (d.true_egfr + assay_bias).round()          # what the lab reports
    return d

dev = site(6000, seed=1)
model = LogisticRegression(max_iter=1000).fit(dev[predictors], dev.outcome)

def evaluate(name, d):
    p = model.predict_proba(d[predictors])[:, 1]
    recal = LogisticRegression(penalty=None).fit(np.log(p / (1 - p)).reshape(-1, 1), d.outcome)
    print(f"{name:<28} n={len(d):>5}  outcome {d.outcome.mean():5.1%}  "
          f"AUROC {roc_auc_score(d.outcome, p):.3f}  predicted {p.mean():5.1%}  "
          f"slope {recal.coef_[0][0]:.2f}")

evaluate("Internal (same hospital)", site(3000, seed=2))
evaluate("Older population", site(3000, seed=3, age_mean=70))
evaluate("Different eGFR assay (+15)", site(3000, seed=4, assay_bias=15))
evaluate("Lower baseline risk", site(3000, seed=5, baseline=-2.8))

# Internal (same hospital)     n= 3000  outcome 18.7%  AUROC 0.734  predicted 18.8%  slope 0.90
# Older population             n= 3000  outcome 23.0%  AUROC 0.757  predicted 23.5%  slope 1.02
# Different eGFR assay (+15)   n= 3000  outcome 18.4%  AUROC 0.759  predicted 12.0%  slope 1.03
# Lower baseline risk          n= 3000  outcome 11.3%  AUROC 0.759  predicted 19.4%  slope 0.99

### Small validation samples are uncertain

Continuing from the previous block: bootstrap confidence interval for AUROC in a 400-patient sample.

In [ ]:
small = site(400, seed=6)
p = model.predict_proba(small[predictors])[:, 1]
rng = np.random.default_rng(0)
boot = []
for _ in range(1000):
    i = rng.integers(0, len(small), len(small))
    if small.outcome.iloc[i].nunique() == 2:
        boot.append(roc_auc_score(small.outcome.iloc[i], p[i]))
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"External sample of {len(small)} patients, {small.outcome.sum()} events: "
      f"AUROC {roc_auc_score(small.outcome, p):.3f} (95% CI {lo:.3f} to {hi:.3f})")

# External sample of 400 patients, 84 events: AUROC 0.750 (95% CI 0.690 to 0.807)

## Exercise

**Goal:** Explore how different kinds of dataset shift affect a clinical prediction model, and fix what can be fixed.

**Data:** Synthetic sites generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the four-site comparison and draw calibration plots for each site.

In [ ]:
# Your code here


### Step 2

Add a concept-shift site where albuminuria matters half as much, and check AUROC and calibration.

In [ ]:
# Your code here


### Step 3

Recalibrate the model at the assay-shift and low-risk sites using 500 local patients, and evaluate on the rest.

In [ ]:
# Your code here


### Step 4

Repeat the bootstrap for samples of 200, 400 and 2,000 patients and plot how the interval narrows.

In [ ]:
# Your code here


### Step 5

Write the validation plan you would require before deploying this model at a new hospital.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/evaluation-safety/external-validation-dataset-shift](https://ramkumarmd.com/medical-ai/evaluation-safety/external-validation-dataset-shift). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.